# Анализ погодных данных (финал пайплайна)

Пайплайн: **генератор → ClickHouse → Jupyter**. Подключаемся к ClickHouse и по показаниям метеостанций смотрим суточный и сезонный профиль, микроклимат разных городов и аномалии.

In [ ]:
%matplotlib inline
import os
import clickhouse_connect
import pandas as pd
import matplotlib.pyplot as plt

# Jupyter работает в compose-сети: хост ClickHouse = имя сервиса.
client = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "clickhouse"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    username=os.environ.get("CLICKHOUSE_USER", "default"),
    password=os.environ.get("CLICKHOUSE_PASSWORD", "clickhouse"),
)
db = os.environ.get("CLICKHOUSE_DB", "weather")
print("Подключено к ClickHouse")

In [ ]:
rows = client.query(
    f"SELECT count(), countDistinct(station_id), min(timestamp), max(timestamp) FROM {db}.readings"
).result_rows[0]
print(f"Показаний: {rows[0]:,}".replace(",", " "))
print(f"Станций: {rows[1]}")
print(f"Диапазон: {rows[2]}  …  {rows[3]}")

## 1. Суточный профиль

Посмотрим среднюю температуру по часам за последние 24 часа (окно считается от последнего замера в базе — так ноутбук работает даже на «устаревших» данных).

In [ ]:
sql = f"""
SELECT toStartOfHour(timestamp) AS hour,
       round(avg(temperature_c), 2) AS avg_temp
FROM {db}.readings
WHERE timestamp >= (SELECT max(timestamp) FROM {db}.readings) - INTERVAL 24 HOUR
GROUP BY hour ORDER BY hour
"""
df = client.query_df(sql)
df["hour"] = pd.to_datetime(df["hour"])

plt.figure(figsize=(12, 4))
plt.plot(df["hour"], df["avg_temp"], marker="o", linewidth=2)
plt.title("Средняя температура по часам (последние 24 ч)")
plt.ylabel("°C")
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

**Что видно:** у температуры есть выраженный суточный цикл — днём теплее, ночью холоднее. Этот ритм заложен генератором, но он же характерен и для реальных метеоданных. Для аналитика важно не «сгладить» такой цикл случайным шумом — иначе суточная динамика потеряется.

## 2. Сезонный профиль

Теперь агрегируем температуру по месяцам за весь период. Один запрос — и виден годовой ход.

In [ ]:
season = client.query_df(f"""
    SELECT toStartOfMonth(timestamp) AS month,
           round(avg(temperature_c), 2) AS avg_temp
    FROM {db}.readings
    GROUP BY month ORDER BY month
""")
season["month"] = pd.to_datetime(season["month"])

plt.figure(figsize=(12, 4))
plt.plot(season["month"], season["avg_temp"], linewidth=2, color="tab:orange")
plt.title("Сезонный ход температуры (средняя за месяц)")
plt.ylabel("°C")
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

**Что видно:** яркий годовой цикл — зимой холодно, летом тепло. Такая сезонность — характерная черта климата и важный фон для поиска аномалий: любое отклонение (например, «теплая зима») имеет смысл оценивать относительно ожидаемого сезонного уровня, а не абсолютных значений.

## 3. Микроклимат по городам

Сравним города через джойн факта со справочником станций. Строим среднюю температуру по месяцам для каждого города.

In [ ]:
city = client.query_df(f"""
    SELECT s.city,
           toStartOfMonth(r.timestamp) AS month,
           round(avg(r.temperature_c), 2) AS avg_temp
    FROM {db}.readings r
    LEFT JOIN {db}.stations s ON r.station_id = s.station_id
    GROUP BY s.city, month ORDER BY month
""")
city["month"] = pd.to_datetime(city["month"])
wide = city.pivot(index="month", columns="city", values="avg_temp")

wide.plot(figsize=(12, 4), linewidth=2)
plt.title("Средняя температура по городам")
plt.ylabel("°C")
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

**Что видно:** города «живут» в разном климате — южные теплее северных, амплитуда и средний уровень отличаются. Это иллюстрирует силу star-схемы: факт хранит измерения, справочник — атрибуты станции (город), а джойн по ключу `station_id` даёт нужный срез без дублирования данных.

In [ ]:
# Аномалии по дням
anom = client.query_df(f"""
    SELECT toDate(timestamp) AS day, sum(is_anomaly) AS anomalies
    FROM {db}.readings
    WHERE is_anomaly = 1
    GROUP BY day ORDER BY day
""")
anom["day"] = pd.to_datetime(anom["day"])

plt.figure(figsize=(12, 3))
plt.bar(anom["day"], anom["anomalies"], color="tab:red")
plt.title("Аномалии по дням")
plt.ylabel("кол-во")
plt.tight_layout()
plt.show()

**Итог.** Мы прошли весь цикл: данные льются из генератора в ClickHouse, а Jupyter позволяет быстро строить срезы и графики без отдельного BI. Дальше такие же витрины можно вынести в **материализованные представления** (агрегаты по часам/дням), чтобы дашборды в Superset читали уже готовые суммы, а не сканировали миллионы строк каждый раз.